# Laboratorium 1: Powtórzenie, podstawowe przetwarzanie danych

**Kurs:** Programowanie w Pythonie

**Data:** 24.10.2026

**Zakres:** podstawowe wczytywanie danych, filtrowanie, podsumowywanie i raportowanie

## Scenariusz
Tworzysz pierwszą wersję narzędzia do przetwarzania danych z eksperymentów i obserwacji. Każdy wiersz reprezentuje jedną odpowiedź zarejestrowaną podczas eksperymentu. W tym laboratorium wczytasz dane, sprawdzisz ich strukturę, przefiltrujesz rekordy, przygotujesz podsumowanie i utworzysz prosty raport.

## Cele
Po zakończeniu laboratorium potrafisz:
- przeanalizować strukturę niewielkiego zbioru danych;
- wykorzystać `list`, `dictionary` i `set` do reprezentowania oraz przeglądania rekordów;
- pisać funkcje z jasno określonymi parametrami (parameters) i wartościami zwrotnymi (return values);
- wczytać dane CSV i przekształcić wybrane pola;
- filtrować i agregować obserwacje;
- oddzielić obliczenia od prezentacji;
- świadomie obsłużyć pustą kolekcję i brak pliku.

> **Zasada pracy:** przed uruchomieniem kodu spróbuj przewidzieć wynik jego działania - jeśli różni się on od przewidywania, zastanów się, z czego wynikała różnica.


## 0. Konfiguracja

Umieść notebook w tym samym katalogu co następujące pliki:
- `lab01_observations.csv`
- `lab01_observations.json`
- `lab01_malformed.csv`

Uruchom następną komórkę, aby zdefiniować ścieżki. Jeżeli pliki znajdują się w innym miejscu, zmień `DATA_DIR`.


In [9]:
from pathlib import Path

DATA_DIR = Path('.')
CSV_PATH = DATA_DIR / 'lab01_observations.csv'
JSON_PATH = DATA_DIR / 'lab01_observations.json'
MALFORMED_PATH = DATA_DIR / 'lab01_malformed.csv'

print('CSV exists:', CSV_PATH.exists())
print('JSON exists:', JSON_PATH.exists())

CSV exists: True
JSON exists: True


## 1. Zrozumienie danych

Przed napisaniem kodu zapoznajmy się z danymi. Sprawdź pierwsze wiersze pliku CSV i odpowiedz:
1. Co reprezentuje jeden wiersz?
2. Które kolumny zawierają:
- identyfikatory,
- wartości kategoryczne,
- wartości numeryczne,
- wartości logiczne (Boolean)?
3. Które wartości trzeba będzie przekształcić po wczytaniu CSV?
4. Podaj trzy przykładowe pytania, na które można odpowiedzieć przy użyciu tego zbioru.

Zapisz odpowiedzi w poniższej komórce.

**Twoje odpowiedzi:**

1. ...
2. ...
3. ...
4. ...


## 2. Rozgrzewka

Uruchom poniższą komórkę, zawierającą przykładowe dane. Następnie napisz wyrażenia, które:
- wyodrębniają wszystkie identyfikatory uczestników (list comprehension);
- tworzą `set` unikalnych indentyfikatorów uczestników;
- tworzą `list` czasów reakcji;
- tworzą `list` zawierającą wyłącznie poprawne obserwacje.

In [1]:
sample_observations = [
    {'participant_id': 'P001', 'group': 'control', 'response_time_ms': 842, 'correct': True},
    {'participant_id': 'P001', 'group': 'control', 'response_time_ms': 1105, 'correct': False},
    {'participant_id': 'P002', 'group': 'experimental', 'response_time_ms': 756, 'correct': True},
]

print(sample_observations)

[{'participant_id': 'P001', 'group': 'control', 'response_time_ms': 842, 'correct': True}, {'participant_id': 'P001', 'group': 'control', 'response_time_ms': 1105, 'correct': False}, {'participant_id': 'P002', 'group': 'experimental', 'response_time_ms': 756, 'correct': True}]


In [23]:
# TODO: napisz wyrażenia:
participant_ids = [el['participant_id'] for el in sample_observations]
unique_participants = set(participant_ids)
response_times = [el['response_time_ms'] for el in sample_observations]
correct_observations = [el for el in sample_observations if el['correct']]

print(participant_ids)
print(unique_participants)
print(response_times)
print(correct_observations)

['P001', 'P001', 'P002']
{'P002', 'P001'}
[842, 1105, 756]
[{'participant_id': 'P001', 'group': 'control', 'response_time_ms': 842, 'correct': True}, {'participant_id': 'P002', 'group': 'experimental', 'response_time_ms': 756, 'correct': True}]


### 2.1 Napisz funkcję podsumowującą

Zaimplementuj `summarize_numbers(values)`, która zwraca `dictionary` z keys `count`, `minimum`, `maximum` i `mean`. Zdecyduj, co ma zostać zwrócone dla pustej listy, i opisz tę decyzję. Nie używaj `print` wewnątrz funkcji.

In [19]:
def summarize_numbers(values):
    # TODO
    if len(values) ==0:
        return {'count': 0, 'minimum': None, 'maximum': None, 'mean':None}
    return {'count': len(values), 'minimum': len(values), 'maximum': len(values), 'mean':sum(values)/len(values)}
assert summarize_numbers([2, 4, 6])['mean'] == 4
# TODO: Dodaj analogiczne sprawdzenie dla pustej listy

### 2.2 Wybór reprezentacji: `list` czy `tuple`?
Struktura danych opisująca wyniki powinna zawierać trzy elementy: participant ID (string), liczba obserwacji (int) i średni czas reakcji (float).

1. Zdefiniuj strukturę z tymi trzema elementami jako `tuple`.
2. Przedstaw te same informacje jako `list`.
3. Wyjaśnij w Markdown cell, które rozwiązanie jest lepsze, jeżeli struktura nie powinna się zmieniać.


In [28]:
# TODO: utwórz obie struktury danych
summary_tuple = ("id01",7,100,67)
summary_list = ["id01",7,100,67]

# TODO: dokonaj inspekcji, spróbuj zmodyfikować wartości
print(type(summary_tuple), type(summary_list))
summary_list[1] = 4
print (summary_list)

<class 'tuple'> <class 'list'>
['id01', 4, 100, 67]


**Twoje wyjaśnienie:** ...


### 2.3 Normalizacja wartości tekstowych
Znormalizuj poniższe teksty: usuń białe znaki (whitespace) na początku i końcu oraz zmień litery na małe (lowercase). Następnie zastąp spacje znakiem podkreślenia.

Wskazówka: list comprehension.

In [6]:
raw_labels = [' Control ', 'EXPERIMENTAL', ' Pilot Group ']
# TODO: normalizacja
normalized_labels = [el.strip().lower() for el in raw_labels]
normalized_labels

['control', 'experimental', 'pilot group']

### 2.4 Built-in summaries i ordering
Korzystając z `response_times` z początku punktu 2, oblicz minimum i maximum oraz utwórz listę z posortowanymi wartościami. Zaokrąglij średnią do dwóch miejsc po przecinku.


In [29]:
# TODO
minimum_time = min(response_times)
maximum_time = max(response_times)
sorted_times = sorted(response_times)
rounded_mean = sum(response_times)/len(rosponse_times),2
print(minimum_time, maximum_time, sorted_times, rounded_mean)

<class 'NameError'>: name 'rosponse_times' is not defined

## 3. Wczytywanie i konwersja danych CSV

Zaimplementuj `parse_boolean` i `load_observations`. Użyj `csv.DictReader`, context manager, encoding UTF-8 oraz `newline=''`. Przekształć IDs i response times do `int`, a correctness do `bool`.

Na tym etapie walidacja ma pozostać prosta.

In [35]:
import csv


def parse_boolean(value):
    # TODO: zwróć wartość True tylko, jeśli tekst wejściowy to 'true'; nie zwracaj uwagi na rozmiar liter czy białe znaki
    pass


def load_observations(path):
    observations = []
    # TODO:
    # 1) otwórz ścieżkę (with open(...) as source:)
    with open(path, mode='r', encoding='utf-8', newline='') as source:
    # 2) zdefiniuj reader do wczytywania danych z pliku csv (reader = csv.DictReader(...))
    reader = csv.DictReader(source)    
    # 3) przeiteruj przez dane
    for el in reader:
    # 4) dla każdego wczytanego słownika odpowiednio przekonwertuj dane na odpowiedni typ
    el['observation_id'] = int(el['observation_id'])
    el['response_time_ms'] = int(el['response_time_ms'])
    el['correct'] = parse_boolean(el['correct'])
    # 5) dodaj przekonwertowany słownik do listy wyników
    # 6) zwróć listę wyników
    
load_observations(CSV_PATH)

observations = load_observations(CSV_PATH)
print('Loaded:', len(observations))
print('First record:', observations[0])
print('Value types:', {key: type(value).__name__ for key, value in observations[0].items()})

<class 'IndentationError'>: expected an indented block after 'with' statement on line 13 (<ipython-input-35-4af7ad6a646c>, line 15)

### 3.1 Sprawdzenie wczytanego zbioru danych

Oblicz i wyświetl liczbę rekordów, pierwszy rekord, liczbę unikalnych uczestników oraz wszystkie unikalne grupy.


In [ ]:
# TODO
record_count = len(observations)
first_record = observations(0)
participant_count = len(set([el['participant_id'] for el in observations]))
groups = list(set([el['group'] for el in observations]))

print(record_count, first_record, participant_count, groups)

### 3.2 Zachowanie w przypadku brakującego pliku (missing file)

Wywołaj loader dla ścieżki, który nie istnieje. Przechwyć wyłącznie błąd `FileNotFoundError` i wyświetl użyteczny komunikat. Nie dodawaj ogólnego bloku `except Exception`.

In [38]:
missing_path = DATA_DIR / 'does_not_exist.csv'

# TODO: wywołaj funkcję load_observations wewnątrz bloku 'try', obsłuż 'except FileNotFoundError'
try:
    load_observations(missing_path)
except: FileNotFoundError.csv

<class 'AttributeError'>: type object 'FileNotFoundError' has no attribute 'csv'

## 4. Filtrowanie i transformacja danych

Zaimplementuj poniższe funkcje. Każda powinna zwracać nową listę i pozostawiać listę wejściową bez zmian.


In [42]:
def filter_by_group(observations, group):
    # TODO
    return [el for el in observations if el['group']==group]


def filter_correct(observations):
    # TODO
    return [el for el in observations if el['correct']]


def filter_by_response_time(observations, minimum_ms=None, maximum_ms=None):
    # TODO
    # W przypadku, gdy wartość minimum_ms bądź maximum_ms nie jest równa None, obserwacje powinny być sprawdzane względem nich
    # Sprawdzenie może następować względem jednej wartości (jedna pozostaje None), bądź żadnej z wartości (obie None)
    result = []
    for el in observations:
        response_time = el['response_time_ms']
        if minimum_ms is not None and response_time < minimum_ms:
            continue
        if maximum_ms is not None and response_time > maximum_ms:

control = filter_by_group(observations, 'control')
correct = filter_correct(observations)
within_range = filter_by_response_time(observations, minimum_ms=700, maximum_ms=1000)
print(len(control), len(correct), len(within_range))

<class 'IndentationError'>: expected an indented block after 'if' statement on line 20 (<ipython-input-42-a0b18f7846fb>, line 22)

### 4.1 Wybór prostszej reprezentacji

Utwórz nową `list` obiektów `dictionary`, zawierających wyłącznie `participant_id`, `response_time_ms` i `correct`. Sprawdź, czy każdy otrzymany `dictionary` ma dokładnie te same klucze.


In [ ]:
# TODO
simplified = ...

assert all(set(item) == {'participant_id', 'response_time_ms', 'correct'} for item in simplified)
simplified[:3]

## 5. Agregacja danych

Zaimplementuj poniższe funkcje. Dla pustych danych wejściowych zwróć `None`. Zrób podobnie, jeżeli wartości `mean` lub `accuracy` są niedostępne.


In [ ]:
def calculate_accuracy(observations):
    # TODO
    pass


def calculate_mean_response_time(observations):
    # TODO
    pass


def count_by_participant(observations):
    # TODO
    pass

assert calculate_accuracy([]) is None
assert calculate_mean_response_time([]) is None
print(calculate_accuracy(observations))
print(calculate_mean_response_time(observations))
print(count_by_participant(observations))

### 5.1 Tworzenie podsumowania na poziomie grup

Utwórz zagnieżdżony słownik zawierający liczbę obserwacji, `accuracy` i `mean_response_time` dla każdej group.


In [ ]:
def summarize_by_group(observations):
    # TODO
    pass

group_summary = summarize_by_group(observations)
group_summary

### 5.2 Zliczanie częstotliwości wystąpienia z użyciem `dictionary` i `Counter`
Policz obserwacje według wystąpienia `stimulus` na dwa sposoby:
1. przy użyciu zwykłego `dictionary`;
2. przy użyciu `collections.Counter`.

Sprawdź, czy oba podejścia zwracają identyczne counts.


In [ ]:
# TODO: implementacja przy użyciu słownika
# wskazówka: użyj pętli for
stimulus_counts = ...

from collections import Counter
# TODO: implementacja przy użyciu Counter
# wskazówka: https://docs.python.org/3/library/collections.html#collections.Counter
counter_counts = ...

assert dict(counter_counts) == stimulus_counts
stimulus_counts

### 5.3 Opcjonalnie: porównanie z wartościami ze `statistics.mean`
Użyj `statistics.mean`, aby sprawdzić ręcznie obliczone średnie czasy reakcji. Nie zastępuj pierwotnej funkcji - porównaj dwa podejścia.


In [ ]:
from statistics import mean
# TODO
# wskazówka: https://docs.python.org/3/library/statistics.html#statistics.mean
library_mean = ...
assert library_mean == calculate_mean_response_time(observations)
library_mean

## 6. Tworzenie raportu

Oddziellmy obliczenia od prezentacji danych.

Zaimplementuj `create_report(observations)` tak, aby zwracała jedną wartość tekstową, rozciągająca się przez wiele linii. Uwzględnij: liczbę obserwacji, liczbę osób badanych, `accuracy`, `mean_response_time` zagregowane na poziomie całego zbioru.

Bonus: uwzględnij dodatkowe sekcje dla konkretnych grup (`extend`).


In [ ]:
def format_percentage(value):
    return 'unavailable' if value is None else f'{value:.2%}'


def format_measurement(value):
    return 'unavailable' if value is None else f'{value:.2f} ms'


def create_report(observations):
    # TODO:
    # 1) oblicz wymagane wartości
    # 2) skonstruuj listę zawierającą tekstowy opis wymaganych danych (np. f'Number of observations: {}')
    # 3) zwróć tekst ('\n'.join(lines))
    pass

report = create_report(observations)
print(report)

## 7. Weryfikacja

Uruchom poniższe sprawdzenia. Dodaj co najmniej dwa własne. Nieudany `assert` wskazuje elementy wymagające poprawy.

In [ ]:
assert len(observations) == 24
assert len({item['participant_id'] for item in observations}) == 8
assert set(item['group'] for item in observations) == {'control', 'experimental'}
assert calculate_accuracy([]) is None
assert filter_by_group(observations, 'unknown') == []

# TODO: dodaj przynajmniej dwa własne sprawdzenia
print('Core checks passed.')

## 8. Zadania dodatkowe

1. **JSON input:** zaimplementuj `load_json_observations(path)`, zachowując tę samą reprezentację jak w przypadku danych wczytanych z .csv.
2. **Zniekształcone dane wejściowe:** spróbuj wczytać `lab01_malformed.csv`. Zapisz exception type i wiadomość. Obsłuż wyjątek (nie przy pomocy ogólnego `except`).
3. **Zapisanie raportu:** zapisz zwrócony raport jako `lab01_report.txt` (encoding: UTF-8).

## 9. Podsumowanie

Odpowiedz na następujące pytania:
- Które zadanie zajęło najwięcej czasu?

- Które zadanie było najtrudniejsze?

- Które koncepty były najtrudniejsze do zrozumienia?

- Która część programu stałaby się trudna w obsłudze wraz ze wzrostem dataset?

- Co należałoby zmienić, gdyby dane wejściowe były niepoprawne?